In [ ]:
import json
import pandas as pd

with open("../../dataset/train_sentiment.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df1 = pd.DataFrame(data)

# print(df.head())

In [ ]:
df2 = pd.read_csv('../../dataset/review_shopping.csv', sep='\t', names=['text', 'sentiment'], header=None)
df2['sentiment'] = df2['sentiment'].replace({
    'neg': 'negative',
    'pos': 'positive'
})

In [ ]:
df = pd.concat([df1], ignore_index=True)
df['sentiment'].value_counts().plot.bar()

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../"))

In [ ]:
from models.preprocess import clean_text
text = df['text']
sentiment = df['sentiment']

# text = df["text"].apply(clean_text)
text.head()

In [ ]:
from sklearn.model_selection import train_test_split

text_train, text_val, sentiment_train, sentiment_val = train_test_split(
    text, sentiment, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from models.preprocess import tokenizer

vectorizer = TfidfVectorizer(
    tokenizer=tokenizer,
    ngram_range=(1,2),
)

text_train_tfidf = vectorizer.fit_transform(text_train)
text_val_tfidf = vectorizer.transform(text_val)

In [ ]:
from sklearn.svm import LinearSVC

model = LinearSVC(C=1.5)

model.fit(text_train_tfidf, sentiment_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report # type: ignore

y_pred = model.predict(text_val_tfidf)

print("Accuracy:", accuracy_score(sentiment_val, y_pred))
print(classification_report(sentiment_val, y_pred))

In [ ]:
import pickle

with open("../../models/SVM/sentiment_model.pkl", "wb") as f:
    pickle.dump((vectorizer, model), f)

In [ ]:
import pickle

with open("sentiment_model.pkl", "rb") as f:
    vectorizer, model = pickle.load(f)

test_text = "ไป ตาย ไป"

vec = vectorizer.transform([test_text])
prediction = model.predict(vec)

print("Prediction:", prediction[0])